# 인용 참고문헌 arXiv 메타데이터 수집 (OAI-PMH)

`data/citations_ai/arxiv_citations_part1.jsonl`, `part2.jsonl`의 `references[].arxiv_id`를 수집합니다. 결과의 `arxiv_id`는 **참조된 논문**, `cit_arxiv_id`는 그 논문을 인용한 **원본 논문 ID의 리스트**입니다.

arXiv OAI-PMH 요청 간격(3초), 재시도, 원자적 파일 저장, 메타데이터 캐시를 적용했습니다. 최초 전체 수집은 오래 걸릴 수 있으나, 중단 후 다시 실행하면 이미 캐시된 참조 논문은 재조회하지 않습니다.

In [1]:
import hashlib
import json
from pathlib import Path

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / 'pyproject.toml').exists() and (path / 'data' / 'citations_ai').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('프로젝트 루트 또는 notebooks/ 디렉터리에서 실행하세요.')

INPUT_DIR = ROOT / 'data' / 'citations_ai'
INPUT_FILES = [INPUT_DIR / 'arxiv_citations_part1.jsonl', INPUT_DIR / 'arxiv_citations_part2.jsonl']
OUTPUT_DIR = ROOT / 'data' / 'ai_references'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILE_PREFIX = 'arxiv_ai_references_oai'
CHUNK_SIZE = 5_000
METADATA_CACHE_PATH = OUTPUT_DIR / f'{FILE_PREFIX}_metadata_cache.jsonl'
STATE_PATH = OUTPUT_DIR / f'{FILE_PREFIX}_state.json'

missing_files = [path for path in INPUT_FILES if not path.exists()]
if missing_files:
    raise FileNotFoundError(f'입력 파일이 없습니다: {missing_files}')

print(f'입력: {[path.name for path in INPUT_FILES]}')
print(f'출력: {OUTPUT_DIR}')

입력: ['arxiv_citations_part1.jsonl', 'arxiv_citations_part2.jsonl']
출력: c:\Users\Playdata\Desktop\arxiv_graph_RAG\data\ai_references


In [2]:
import json
import re
from pathlib import Path

ARXIV_ID_RE = re.compile(r'^(?:arXiv:)?(?P<id>(?:\d{4}\.\d{4,5}|[A-Za-z-]+(?:\.[A-Za-z-]+)?/\d{7}))(?:v\d+)?$')
OUTPUT_FIELDS = ('id', 'title', 'abstract', 'authors', 'categories', 'primary_category', 'published', 'updated', 'doi', 'pdf_url', 'source')

def normalize_arxiv_id(value):
    if not isinstance(value, str):
        return None
    match = ARXIV_ID_RE.fullmatch(value.strip())
    return match.group('id') if match else None

def collect_references(files):
    """Return one record per referenced paper with unique citing IDs in input order."""
    collected_references, by_arxiv_id, skipped = [], {}, 0
    for path in files:
        with Path(path).open(encoding='utf-8-sig') as handle:
            for line_number, line in enumerate(handle, start=1):
                if not line.strip():
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError as error:
                    raise ValueError(f'{path.name}:{line_number} JSON 파싱 실패') from error
                if not isinstance(row, dict):
                    raise ValueError(f'{path.name}:{line_number} 레코드는 객체여야 합니다.')
                cit_arxiv_id = normalize_arxiv_id(row.get('arxiv_id'))
                if not cit_arxiv_id:
                    raise ValueError(f'{path.name}:{line_number} 원본 arxiv_id가 올바르지 않습니다.')
                reference_items = row.get('references') or []
                if not isinstance(reference_items, list):
                    raise ValueError(f'{path.name}:{line_number} references는 목록이어야 합니다.')
                for reference in reference_items:
                    arxiv_id = normalize_arxiv_id(reference.get('arxiv_id') if isinstance(reference, dict) else None)
                    if not arxiv_id:
                        skipped += 1
                        continue
                    record = by_arxiv_id.get(arxiv_id)
                    if record is None:
                        record = {'arxiv_id': arxiv_id, 'cit_arxiv_id': []}
                        by_arxiv_id[arxiv_id] = record
                        collected_references.append(record)
                    if cit_arxiv_id not in record['cit_arxiv_id']:
                        record['cit_arxiv_id'].append(cit_arxiv_id)
    return collected_references, skipped

def make_output_record(edge, metadata, found):
    record = {'arxiv_id': edge['arxiv_id'], 'cit_arxiv_id': edge['cit_arxiv_id'], 'found': found}
    if metadata is None:
        record.update({
            'id': None, 'title': None, 'abstract': None, 'authors': [], 'categories': [],
            'primary_category': None, 'published': None, 'updated': None, 'doi': None,
            'pdf_url': None, 'source': 'arxiv',
        })
        return record
    record.update({field: metadata.get(field) for field in OUTPUT_FIELDS})
    return record

In [3]:
import http.client
import ssl
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
import truststore

OAI_URL = 'https://export.arxiv.org/oai2'
METADATA_PREFIX = 'arXiv'
REQUEST_INTERVAL_SECONDS = 3.0
MAX_RETRIES = 8
BACKOFF_BASE_SECONDS = 10.0
BACKOFF_MAX_SECONDS = 300.0
RETRYABLE_HTTP_CODES = {429, 500, 502, 503, 504}
RETRYABLE_NETWORK_ERRORS = (urllib.error.URLError, TimeoutError, ConnectionError, http.client.HTTPException)
OAI = 'http://www.openarchives.org/OAI/2.0/'
ARXIV = 'http://arxiv.org/OAI/arXiv/'
NS = {'oai': OAI, 'arxiv': ARXIV}
_last_request_at = 0.0

def _text(node, path):
    child = node.find(path, NS)
    return child.text.strip() if child is not None and child.text else None

def _retry_delay(error, attempt):
    retry_after = getattr(error, 'headers', None) and error.headers.get('Retry-After')
    if retry_after:
        try:
            return max(float(retry_after), REQUEST_INTERVAL_SECONDS)
        except ValueError:
            pass
    return min(BACKOFF_BASE_SECONDS * (2 ** attempt), BACKOFF_MAX_SECONDS)

def _ssl_context():
    return truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT)

def _get_xml(params):
    global _last_request_at
    url = f'{OAI_URL}?{urllib.parse.urlencode(params)}'
    request = urllib.request.Request(url, headers={'User-Agent': 'arxiv-citation-reference-harvester/1.0'})
    for attempt in range(MAX_RETRIES + 1):
        elapsed = time.monotonic() - _last_request_at
        if elapsed < REQUEST_INTERVAL_SECONDS:
            time.sleep(REQUEST_INTERVAL_SECONDS - elapsed)
        try:
            _last_request_at = time.monotonic()
            with urllib.request.urlopen(request, timeout=120, context=_ssl_context()) as response:
                return ET.fromstring(response.read())
        except urllib.error.HTTPError as error:
            if error.code not in RETRYABLE_HTTP_CODES or attempt >= MAX_RETRIES:
                detail = error.read().decode('utf-8', errors='replace')
                error.close()
                raise RuntimeError(f'OAI-PMH HTTP {error.code}: {detail[:300]}') from error
            delay = _retry_delay(error, attempt)
            print(f'HTTP {error.code}: {delay:.0f}초 후 재시도 ({attempt + 1}/{MAX_RETRIES})')
            error.close()
            time.sleep(delay)
        except RETRYABLE_NETWORK_ERRORS as error:
            if attempt >= MAX_RETRIES:
                raise
            delay = min(BACKOFF_BASE_SECONDS * (2 ** attempt), BACKOFF_MAX_SECONDS)
            print(f'네트워크 오류({error!r}): {delay:.0f}초 후 재시도 ({attempt + 1}/{MAX_RETRIES})')
            time.sleep(delay)
    raise RuntimeError('재시도 횟수를 초과했습니다.')

def _parse_oai_record(record):
    header = record.find('oai:header', NS)
    if header is None or header.attrib.get('status') == 'deleted':
        return None
    metadata = record.find('oai:metadata/arxiv:arXiv', NS)
    if metadata is None:
        return None
    identifier = _text(metadata, 'arxiv:id')
    if not identifier:
        return None
    authors = []
    for author in metadata.findall('arxiv:authors/arxiv:author', NS):
        name = f"{_text(author, 'arxiv:forenames') or ''} {_text(author, 'arxiv:keyname') or ''}".strip()
        if name:
            authors.append(name)
    categories = (_text(metadata, 'arxiv:categories') or '').split()
    created = _text(metadata, 'arxiv:created')
    updated = _text(metadata, 'arxiv:updated') or created
    return {
        'id': f'https://arxiv.org/abs/{identifier}',
        'title': ' '.join((_text(metadata, 'arxiv:title') or '').split()),
        'abstract': ' '.join((_text(metadata, 'arxiv:abstract') or '').split()),
        'authors': authors, 'categories': categories, 'primary_category': categories[0] if categories else None,
        'published': f'{created}T00:00:00Z' if created else None,
        'updated': f'{updated}T00:00:00Z' if updated else None,
        'doi': _text(metadata, 'arxiv:doi'), 'pdf_url': f'https://arxiv.org/pdf/{identifier}', 'source': 'arxiv',
    }

def fetch_arxiv_metadata(arxiv_id):
    root = _get_xml({'verb': 'GetRecord', 'identifier': f'oai:arXiv.org:{arxiv_id}', 'metadataPrefix': METADATA_PREFIX})
    error = root.find('oai:error', NS)
    if error is not None:
        if error.attrib.get('code') in {'idDoesNotExist', 'noRecordsMatch'}:
            return None
        raise RuntimeError(f"OAI-PMH {error.attrib.get('code')}: {error.text}")
    record = root.find('oai:GetRecord/oai:record', NS)
    return _parse_oai_record(record) if record is not None else None

In [4]:
import os
import tempfile
from datetime import datetime, timezone

def atomic_write_jsonl(path, rows):
    path = Path(path)
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n', dir=path.parent, prefix=f'.{path.stem}_', suffix='.tmp', delete=False) as handle:
            temp_path = Path(handle.name)
            for row in rows:
                handle.write(json.dumps(row, ensure_ascii=False) + '\n')
        os.replace(temp_path, path)
    except Exception:
        if temp_path is not None:
            temp_path.unlink(missing_ok=True)
        raise

def load_metadata_cache(path):
    cache = {}
    if not path.exists():
        return cache
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f'{path.name}:{line_number} 캐시 JSON 파싱 실패') from error
            arxiv_id = normalize_arxiv_id(row.get('arxiv_id')) if isinstance(row, dict) else None
            if not arxiv_id or not isinstance(row.get('found'), bool):
                raise ValueError(f'{path.name}:{line_number} 캐시 레코드가 올바르지 않습니다.')
            cache[arxiv_id] = {'found': row['found'], 'metadata': row.get('metadata')}
    return cache

def write_output_chunks(records):
    paths = []
    for index, start in enumerate(range(0, len(records), CHUNK_SIZE), start=1):
        path = OUTPUT_DIR / f'{FILE_PREFIX}_part{index}.jsonl'
        atomic_write_jsonl(path, records[start:start + CHUNK_SIZE])
        paths.append(path)
    return paths

references, skipped_references = collect_references(INPUT_FILES)
target_ids = sorted(reference['arxiv_id'] for reference in references)
input_signature = hashlib.sha256('\n'.join(f"{reference['arxiv_id']}\t{','.join(reference['cit_arxiv_id'])}" for reference in references).encode('utf-8')).hexdigest()
cache = load_metadata_cache(METADATA_CACHE_PATH)
todo = [arxiv_id for arxiv_id in target_ids if arxiv_id not in cache]
print(f'고유 참조 arXiv ID: {len(references):,}건 / arXiv ID 없는 references: {skipped_references:,}건')
print(f'캐시됨: {len(cache):,}건 / 이번 조회: {len(todo):,}건')

for index, arxiv_id in enumerate(todo, start=1):
    metadata = fetch_arxiv_metadata(arxiv_id)
    cache[arxiv_id] = {'found': metadata is not None, 'metadata': metadata}
    atomic_write_jsonl(METADATA_CACHE_PATH, [
        {'arxiv_id': key, 'found': value['found'], 'metadata': value['metadata']}
        for key, value in sorted(cache.items())
    ])
    STATE_PATH.write_text(json.dumps({
        'input_signature': input_signature, 'reference_count': len(references), 'target_count': len(target_ids),
        'cached_count': len(cache), 'updated_at': datetime.now(timezone.utc).isoformat(),
    }, ensure_ascii=False), encoding='utf-8')
    if index == len(todo) or index % 25 == 0:
        print(f'[{index:,}/{len(todo):,}] 캐시 저장 완료: {arxiv_id}')

records = [make_output_record(reference, cache[reference['arxiv_id']]['metadata'], cache[reference['arxiv_id']]['found']) for reference in references]
output_paths = write_output_chunks(records)
print(f'결과 저장: {len(records):,}건 / {len(output_paths)}개 파일')

고유 참조 arXiv ID: 31,913건 / arXiv ID 없는 references: 60,549건
캐시됨: 0건 / 이번 조회: 31,913건
[25/31,913] 캐시 저장 완료: 0804.3582
[50/31,913] 캐시 저장 완료: 0904.0589


KeyboardInterrupt: 

In [ ]:
saved_records = []
for path in sorted(OUTPUT_DIR.glob(f'{FILE_PREFIX}_part*.jsonl')):
    with path.open(encoding='utf-8') as handle:
        saved_records.extend(json.loads(line) for line in handle if line.strip())

saved_by_arxiv_id = {row.get('arxiv_id'): row.get('cit_arxiv_id') for row in saved_records}
expected_by_arxiv_id = {reference['arxiv_id']: reference['cit_arxiv_id'] for reference in references}
assert len(saved_records) == len(saved_by_arxiv_id), '출력에 중복된 참조 논문이 있습니다.'
assert saved_by_arxiv_id == expected_by_arxiv_id, '출력 참조 논문 또는 cit_arxiv_id 목록이 입력과 일치하지 않습니다.'
assert all(isinstance(row.get('cit_arxiv_id'), list) and row['cit_arxiv_id'] for row in saved_records), '비어 있거나 리스트가 아닌 cit_arxiv_id가 있습니다.'
assert all(row.get('source') == 'arxiv' for row in saved_records), '출처가 arxiv가 아닌 결과가 있습니다.'

found_count = sum(row['found'] for row in saved_records)
missing_count = len(saved_records) - found_count
print(f'검증 완료: 출력 {len(saved_records):,}건 = 고유 참조 논문 {len(references):,}건')
print(f'OAI-PMH 수집 성공: {found_count:,}건 / 찾지 못함 또는 삭제됨: {missing_count:,}건')
display(saved_records[:3])